# Fine-tune IBM Granite 4.0 Speech on Non-Standard Kenyan English Speech (Noise-Robust)

Fine-tunes `ibm-granite/granite-speech-4.1-2b` on `cdli/kenyan_english_nonstandard_speech_v1.0`
with waveform-level noise augmentation.

**Key Granite-specific differences from the Whisper notebooks:**
- Uses `GraniteSpeechProcessor` for both text and audio — `processor(audio, sampling_rate=sr)` is
  the correct call; do NOT pass `sampling_rate` to `GraniteSpeechFeatureExtractor` directly.
- Model forward pass requires `input_features_mask` (variable-length audio padding mask).
- The data collator builds `input_ids` from the chat-template prompt + transcription and uses
  a causal LM loss — there is no separate encoder/decoder label structure like Whisper.
- `GraniteSpeechForConditionalGeneration` is the correct model class (not `AutoModelForSpeechSeq2Seq`).
- SpecAugment is not supported — noise robustness is handled entirely by waveform augmentation.

**Baseline config matches whisper-large-v3 Run 2 where possible:**
`AUGMENT_PROB=0.4`, `AUG_REVERB=False`, polynomial LR scheduler,
`EARLY_STOPPING_PATIENCE=7`, `USE_BF16=True`.

## Authentication

In [1]:
from huggingface_hub import login
HF_TOKEN = input('Enter HF token: ')
login(token=HF_TOKEN)

## Settings

**Adapt these for each run. Everything else should be left as-is.**

### Directories

In [2]:
import os

LOCAL_STORAGE_DIR = '/jupyter_kernel'
BASE_DIR = os.path.join(LOCAL_STORAGE_DIR, 'trained_models')
os.makedirs(BASE_DIR, exist_ok=True)

# Increment run number for each new run
OUTPUT_DIR = os.path.join(BASE_DIR, 'ibm-granite-speech-4.1-2b-kenyan-english-nonstandard-robust_v1_run1')

print(f'Will write model to: {OUTPUT_DIR}')
if os.path.exists(OUTPUT_DIR):
    raise ValueError('Output directory already exists. Increment run number or delete existing directory.')

Will write model to: /jupyter_kernel/trained_models/ibm-granite-speech-4.1-2b-kenyan-english-nonstandard-robust_v1_run1


ValueError: Output directory already exists. Increment run number or delete existing directory.

### Model and Dataset Settings

In [3]:
GRANITE_MODEL_ID = 'ibm-granite/granite-speech-4.1-2b'

LANGUAGE    = 'en'
DATASET_NAME = 'cdli/kenyan_english_nonstandard_speech_v1.0'

print(f'Model: {GRANITE_MODEL_ID}')
print(f'Dataset: {DATASET_NAME}')

Model: ibm-granite/granite-speech-4.1-2b
Dataset: cdli/kenyan_english_nonstandard_speech_v1.0


### Augmentation Settings

Matches whisper-large-v3 Run 2 validated settings.
`AUGMENT_PROB=0.4` and `AUG_REVERB=False` are the known-best
configuration for this dataset.

In [4]:
USE_WAVEFORM_AUGMENTATION = True

# Validated best for this dataset across whisper-small Runs 1-6
# and whisper-large-v3 Run 2. Do not increase without testing.
AUGMENT_PROB = 0.4

AUG_VOLUME_PERTURB = True   # mild gain jitter, always on
AUG_GAUSSIAN_NOISE = True   # crowd/ambient noise
AUG_GSM_CODEC      = True   # mobile network compression
AUG_REVERB         = False  # disabled — no benefit seen in whisper series

NOISE_LEVEL_MIN = 0.002
NOISE_LEVEL_MAX = 0.01

### Architecture Settings

Granite 4.0 Speech components:
- `model.encoder` — Conformer CTC encoder (10 blocks)
- `model.projector` — Q-former projector (2-layer temporal downsampler)
- `model.language_model` — Granite 2B LLM (includes LoRA adapters)

Full fine-tuning (`UPDATE_LLM=True`) based on whisper-large-v3 Run 2
finding that unfreezing all components was critical for noise robustness.

In [5]:
UPDATE_ENCODER   = True
UPDATE_PROJECTOR = True
UPDATE_LLM       = True

# Partial LLM unfreezing (only used when UPDATE_LLM=False)
# Granite 3.3-2b LLM has 26 transformer layers (0-25)
NUM_LLM_LAYERS_TO_UNFREEZE = 4

### Trainer Settings

LR reduced to 1e-6 vs whisper-large-v3 Run 2's 5e-7 because Granite's
LoRA adapter needs more gradient signal to adapt than Whisper's full
fine-tuned encoder. EARLY_STOPPING_PATIENCE=7 matches whisper series
best practice — patience=3 caused premature stopping in whisper Run 1.

In [7]:
LOGGING_STEPS = 5
SAVE_STEPS    = 50

MAX_EPOCHS = 10
MAX_STEPS  = 1000

LEARNING_RATE     = 1e-6
LR_SCHEDULER_TYPE = 'polynomial'
LR_WARMUP_STEPS   = 100
LR_END            = 1e-8
LR_DECAY_POWER    = 1

WEIGHT_DECAY            = 0.01
EARLY_STOPPING_PATIENCE = 7

BATCH_SIZE      = 8
EVAL_BATCH_SIZE = 8

MAX_GEN_LEN   = 256   # longer than whisper (128) — Granite includes chat template tokens
EVAL_ON_START = True
EVAL_STEPS    = 50

USE_FP16 = False
USE_BF16 = True   # A100 native BF16 — more stable than FP16 for large models

NUM_CHECKPOINTS_TO_STORE = 2

print(f'LR: {LEARNING_RATE} | Scheduler: {LR_SCHEDULER_TYPE} | Max steps: {MAX_STEPS}')
print(f'Early stopping patience: {EARLY_STOPPING_PATIENCE}')
print(f'Batch size: {BATCH_SIZE} | BF16: {USE_BF16}')

LR: 1e-06 | Scheduler: polynomial | Max steps: 1000
Early stopping patience: 7
Batch size: 8 | BF16: True


## Imports and Environment Setup

In [8]:
import os
import random
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import datasets
import evaluate

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

import transformers
print(f'transformers: {transformers.__version__}')  # must be >=4.52.0

from transformers import (
    AutoProcessor,
    GraniteSpeechForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

try:
    import peft
    print(f'peft: {peft.__version__}')  # required for Granite LoRA
except ImportError:
    raise ImportError('Install PEFT: pip install peft --break-system-packages')

# Disable caching — saves disk on Modal volumes
datasets.disable_caching()
print(f'Dataset caching: {datasets.is_caching_enabled()}')

# Avoid thread contention in multiprocessing map
torch.set_num_threads(1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

num_proc = min(32, os.cpu_count())
print(f'CPU workers: {num_proc}')

wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')
transcript_normalizer = BasicTextNormalizer()

transformers: 4.57.6
peft: 0.19.1
Dataset caching: False
Device: cuda
CPU workers: 20


## Waveform Augmentation

Applied to training audio only, before feature extraction.
Simulates Kenyan field conditions: crowd noise, mobile compression, volume variation.

In [9]:
def augment_audio(audio_array: np.ndarray, sample_rate: int, apply_prob: float = 0.4) -> np.ndarray:
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)  # (1, T)

    # 1. Volume perturbation (always applied)
    if AUG_VOLUME_PERTURB:
        gain = random.uniform(0.7, 1.3)
        waveform = waveform * gain

    # 2. Gaussian noise — crowd/ambient
    if AUG_GAUSSIAN_NOISE and random.random() < apply_prob:
        noise_level = random.uniform(NOISE_LEVEL_MIN, NOISE_LEVEL_MAX)
        waveform = waveform + torch.randn_like(waveform) * noise_level

    # 3. GSM codec — mobile network compression
    if AUG_GSM_CODEC and random.random() < apply_prob:
        resample_down = T.Resample(orig_freq=sample_rate, new_freq=8000)
        resample_up   = T.Resample(orig_freq=8000, new_freq=sample_rate)
        waveform = resample_up(resample_down(waveform))

    # 4. Room reverb (disabled: AUG_REVERB=False)
    if AUG_REVERB and random.random() < apply_prob * 0.5:
        reverb_gain   = random.uniform(0.1, 0.25)
        delay_samples = random.randint(int(0.01 * sample_rate), int(0.05 * sample_rate))
        delayed = torch.zeros_like(waveform)
        delayed[:, delay_samples:] = waveform[:, :-delay_samples]
        waveform = waveform + reverb_gain * delayed

    waveform = torch.clamp(waveform, -1.0, 1.0)
    return waveform.squeeze(0).numpy()

## Helper Functions

In [10]:
def count_trainable_parameters(model):
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable, total


def load_dataset_split(dataset_name: str, split: str, limit_to_30_seconds: bool = True):
    ds = datasets.load_dataset(dataset_name, split=split, streaming=False)
    orig_len = len(ds)
    if limit_to_30_seconds:
        ds = ds.filter(lambda ex: ex['audio_length'] <= 30)
        print(f'[{split}] {orig_len} -> {len(ds)} examples (<= 30s)')
    return ds


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    # Safety fixes
    if isinstance(pred_ids, torch.Tensor):
        pred_ids = pred_ids.cpu().numpy()
    if isinstance(label_ids, torch.Tensor):
        label_ids = label_ids.cpu().numpy()
    
    # Replace -100 with pad token for labels
    label_ids = np.where(label_ids == -100, processor.tokenizer.pad_token_id, label_ids)
    
    # Clip invalid token IDs (important safety)
    vocab_size = len(processor.tokenizer)
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
    
    # Decode safely
    try:
        pred_strs = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_strs = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    except Exception as e:
        print(f"Decoding error: {e}")
        return {'wer': 1.0, 'cer': 1.0}
    
    wers, cers = [], []
    for p, l in zip(pred_strs, label_strs):
        p_norm = transcript_normalizer(p)
        l_norm = transcript_normalizer(l)
        wers.append(wer_metric.compute(predictions=[p_norm], references=[l_norm]))
        cers.append(cer_metric.compute(predictions=[p_norm], references=[l_norm]))
    
    wer = float(np.mean([min(1.0, x) for x in wers]))
    cer = float(np.mean([min(1.0, x) for x in cers]))
    
    print(f'WER: {wer:.4f} | CER: {cer:.4f}')
    return {'wer': wer, 'cer': cer}

## Load Processor

`GraniteSpeechProcessor` wraps both the feature extractor and tokenizer.
**Correct usage:** `processor(audio_array, sampling_rate=sr)` — pass sampling_rate
to the processor, NOT to the feature extractor directly.
The feature extractor's `__call__` does not accept `sampling_rate` as a kwarg.

In [11]:
print(f'Loading processor: {GRANITE_MODEL_ID}')
processor = AutoProcessor.from_pretrained(GRANITE_MODEL_ID)
tokenizer = processor.tokenizer

# Keep decoder-only generation padding left-aligned for correct behavior.
if hasattr(tokenizer, 'padding_side'):
    tokenizer.padding_side = 'left'

# Build ASR prompt once — reused in feature extraction functions.
# <|audio|> is Granite's placeholder for where audio embeddings are injected.
_ASR_CHAT   = [{'role': 'user', 'content': '<|audio|>can you transcribe the speech into a written format?'}]
_ASR_PROMPT = tokenizer.apply_chat_template(_ASR_CHAT, tokenize=False, add_generation_prompt=True)
print(f'Processor: {type(processor).__name__}')
print(f'Tokenizer: {type(tokenizer).__name__}')
print(f'ASR prompt (first 120 chars): {repr(_ASR_PROMPT[:120])}...')

Loading processor: ibm-granite/granite-speech-4.1-2b
Processor: GraniteSpeechProcessor
Tokenizer: GPT2TokenizerFast
ASR prompt (first 120 chars): 'USER: <|audio|>can you transcribe the speech into a written format?\n ASSISTANT:'...


In [12]:
# Fix tokenizer for decoder-only causal LM generation
if hasattr(tokenizer, 'padding_side'):
    tokenizer.padding_side = 'left'
    
print(f"Tokenizer padding_side set to: {tokenizer.padding_side}")
print(f"Pad token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")

Tokenizer padding_side set to: left
Pad token: <|pad|> (id: 100256)


## Feature Extraction Functions

Granite's feature extraction produces two outputs that the model needs:
- `input_features` — log-mel spectrogram for the conformer encoder
- `input_features_mask` — padding mask for variable-length audio

Both are returned when calling `processor(audio, sampling_rate=sr)`.

Labels are built as: `prompt_tokens + transcription_tokens`, masked so
the loss is only computed on the transcription portion.

In [13]:
def _extract_features(audio_array: np.ndarray, sample_rate: int, transcription: str) -> dict:
    """
    Core feature extraction shared by clean and augmented paths.
    Returns input_features, input_features_mask, input_ids, labels.
    """
    # Audio features — call the processor with text and audio but avoid passing sampling_rate
    # to prevent kwargs being forwarded to the tokenizer.
    audio_out = processor(text=_ASR_PROMPT, audio=audio_array, return_tensors=None)
    # Processor may return Python lists for raw arrays, so normalize them to numpy arrays.
    input_features = np.asarray(audio_out['input_features'][0], dtype=np.float32)
    input_features_mask = np.asarray(audio_out['input_features_mask'][0], dtype=np.bool_)

    # Prompt tokens (no loss) must match the processor's expansion of the audio token.
    prompt_ids_raw = audio_out['input_ids'][0]
    prompt_ids = prompt_ids_raw.tolist() if hasattr(prompt_ids_raw, 'tolist') else prompt_ids_raw

    # Transcription tokens (loss computed here)
    transcription_ids = tokenizer(
        transcription + tokenizer.eos_token,
        add_special_tokens=False,
    ).input_ids

    input_ids = prompt_ids + transcription_ids

    # Labels: -100 masks prompt tokens so loss is transcription-only
    labels = [-100] * len(prompt_ids) + transcription_ids

    return {
        'input_features':      input_features,
        'input_features_mask': input_features_mask,
        'input_ids':           input_ids,
        'labels':              labels,
        'token_length':        len(transcription_ids),
    }


def prepare_features(example):
    """Clean feature extraction. Use for dev and test splits."""
    result = _extract_features(
        example['audio']['array'],
        example['audio']['sampling_rate'],
        example['transcription'],
    )
    for k, v in result.items():
        example[k] = v
    return example


def prepare_features_augmented(example):
    """Waveform augmentation then feature extraction. Use for train split only."""
    audio_array = example['audio']['array']
    sample_rate = example['audio']['sampling_rate']

    if USE_WAVEFORM_AUGMENTATION:
        audio_array = augment_audio(audio_array, sample_rate, apply_prob=AUGMENT_PROB)

    result = _extract_features(audio_array, sample_rate, example['transcription'])
    for k, v in result.items():
        example[k] = v
    return example

## Data Collator

Granite is a causal LM — the collator pads `input_ids` and `labels`
to the same length, and pads `input_features` / `input_features_mask`
for variable-length audio. There is no decoder_start_token_id stripping.

In [14]:
@dataclass
class DataCollatorGraniteSpeech:
    """
    Collator for GraniteSpeechForConditionalGeneration with Seq2SeqTrainer.
    Handles padding for:
      - input_features      (audio log-mel spectrogram)
      - input_features_mask (audio padding mask)
      - input_ids           (prompt + transcription tokens)
      - labels              (transcription tokens only, prompt masked with -100)
    """
    processor: Any
    pad_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # ── Audio features ────────────────────────────────────────────────────
        audio_features = [np.asarray(f['input_features'], dtype=np.float32) for f in features]
        audio_masks     = [np.asarray(f['input_features_mask'], dtype=np.bool_) for f in features]

        max_audio_len = max(feat.shape[0] for feat in audio_features)
        max_mask_len  = max(mask.shape[0] for mask in audio_masks)
        feat_dim      = audio_features[0].shape[1]  # 160

        padded_features = np.zeros((len(features), max_audio_len, feat_dim), dtype=np.float32)
        padded_masks    = np.zeros((len(features), max_mask_len), dtype=np.bool_)

        for i, (feat, mask) in enumerate(zip(audio_features, audio_masks)):
            feat_len = feat.shape[0]
            mask_len = mask.shape[0]
            padded_features[i, :feat_len] = feat
            padded_masks[i, :mask_len]    = mask

        # ── Text sequences (input_ids and labels) ─────────────────────────────
        max_text_len = max(len(f['input_ids']) for f in features)

        padded_input_ids = np.full((len(features), max_text_len), self.pad_token_id, dtype=np.int64)
        padded_labels    = np.full((len(features), max_text_len), -100, dtype=np.int64)
        attention_mask   = np.zeros((len(features), max_text_len), dtype=np.int64)

        for i, f in enumerate(features):
            l = len(f['input_ids'])
            padded_input_ids[i, :l] = f['input_ids']
            padded_labels[i, :l]    = f['labels']
            attention_mask[i, :l]   = 1

        return {
            'input_features':      torch.tensor(padded_features),
            'input_features_mask': torch.tensor(padded_masks, dtype=torch.bool),
            'input_ids':           torch.tensor(padded_input_ids),
            'attention_mask':      torch.tensor(attention_mask),
            'labels':              torch.tensor(padded_labels),
        }


## Load and Prepare Datasets

In [15]:
train_dataset = load_dataset_split(DATASET_NAME, split='train', limit_to_30_seconds=True)
train_dataset = train_dataset.map(
    prepare_features_augmented,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc,
)
print(f'Train examples: {len(train_dataset)}')

# Validate dataset consistency: audio token count must match audio feature mask sum.
def verify_granite_dataset(dataset, name):
    audio_token_id = tokenizer.audio_token_id if hasattr(tokenizer, 'audio_token_id') else tokenizer.convert_tokens_to_ids('<|audio|>')
    sample = dataset[0]
    input_ids = np.asarray(sample['input_ids'], dtype=np.int64)
    mask = np.asarray(sample['input_features_mask'], dtype=np.bool_)
    audio_tokens = int((input_ids == audio_token_id).sum())
    mask_sum = int(mask.sum())
    print(f'{name}: audio_tokens={audio_tokens}, mask_sum={mask_sum}, input_ids_len={len(input_ids)}, mask_shape={mask.shape}')
    if audio_tokens != mask_sum:
        raise ValueError(
            f'{name} dataset is inconsistent: audio token count ({audio_tokens}) does not match input_features_mask sum ({mask_sum}).\n'
            'Re-run the feature extraction cells to rebuild the dataset with the updated prompt/audio handling.'
        )

verify_granite_dataset(train_dataset, 'train_dataset')

Filter:   0%|          | 0/4378 [00:00<?, ? examples/s]

[train] 4378 -> 4243 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/4243 [00:00<?, ? examples/s]

Train examples: 4243
train_dataset: audio_tokens=279, mask_sum=279, input_ids_len=317, mask_shape=(279,)


In [16]:
dev_dataset = load_dataset_split(DATASET_NAME, split='validation', limit_to_30_seconds=True)
dev_dataset = dev_dataset.map(
    prepare_features,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc,
)
print(f'Dev examples: {len(dev_dataset)}')

Filter:   0%|          | 0/542 [00:00<?, ? examples/s]

[validation] 542 -> 542 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/542 [00:00<?, ? examples/s]

Dev examples: 542


In [17]:
test_dataset = load_dataset_split(DATASET_NAME, split='test', limit_to_30_seconds=True)
test_dataset = test_dataset.map(
    prepare_features,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc,
)
print(f'Test examples: {len(test_dataset)}')

Filter:   0%|          | 0/928 [00:00<?, ? examples/s]

[test] 928 -> 926 examples (<= 30s)


Map (num_proc=20):   0%|          | 0/926 [00:00<?, ? examples/s]

Test examples: 926


## Load and Configure Model

In [18]:
print(f'Loading model: {GRANITE_MODEL_ID}')
USE_BF16 = globals().get('USE_BF16', False)
base_model = GraniteSpeechForConditionalGeneration.from_pretrained(
    GRANITE_MODEL_ID,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
)
base_model = base_model.to(device)
base_model.config.use_cache = False  # required for gradient checkpointing
print('Model loaded.')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: ibm-granite/granite-speech-4.1-2b


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded.


In [19]:
# VRAM check — run after model load, before Trainer initialization
import torch
allocated = torch.cuda.memory_allocated(0) / 1e9
reserved  = torch.cuda.memory_reserved(0) / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:       {torch.cuda.get_device_name(0)}')
print(f'Allocated: {allocated:.2f} GB / {total:.1f} GB')
print(f'Reserved:  {reserved:.2f} GB / {total:.1f} GB')
print(f'Headroom:  {total - reserved:.2f} GB')

if (total - reserved) < 10:
    print('WARNING: less than 10 GB headroom. Drop BATCH_SIZE to 4 before training.')
elif (total - reserved) < 15:
    print('CAUTION: moderate headroom. Monitor VRAM during first training step.')
else:
    print('OK: sufficient headroom at current BATCH_SIZE.')

GPU:       NVIDIA A100-SXM4-80GB
Allocated: 4.63 GB / 85.1 GB
Reserved:  4.63 GB / 85.1 GB
Headroom:  80.46 GB
OK: sufficient headroom at current BATCH_SIZE.


### Layer Freezing

Run the **full unfreeze** cell (UPDATE_LLM=True) for Run 1.
The partial unfreeze cell is available for future runs if needed.

In [20]:
# Full unfreeze — run this when UPDATE_LLM=True
UPDATE_LLM = globals().get('UPDATE_LLM', True)
UPDATE_ENCODER = globals().get('UPDATE_ENCODER', True)
UPDATE_PROJECTOR = globals().get('UPDATE_PROJECTOR', True)

if UPDATE_LLM:
    model_root = getattr(base_model, 'model', base_model)

    encoder = getattr(model_root, 'encoder', None)
    projector = getattr(model_root, 'projector', None)
    language_model = getattr(model_root, 'language_model', None)

    if encoder is not None:
        encoder.requires_grad_(UPDATE_ENCODER)
    if projector is not None:
        projector.requires_grad_(UPDATE_PROJECTOR)
    if language_model is not None:
        language_model.requires_grad_(True)

    print(f'Encoder trainable:    {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}' if encoder is not None else 'Encoder trainable:    N/A')
    print(f'Projector trainable:  {sum(p.numel() for p in projector.parameters() if p.requires_grad):,}' if projector is not None else 'Projector trainable:  N/A')
    print(f'LLM trainable:        {sum(p.numel() for p in language_model.parameters() if p.requires_grad):,}' if language_model is not None else 'LLM trainable:        N/A')


Encoder trainable:    440,168,796
Projector trainable:  35,697,664
LLM trainable:        1,837,275,136


## If the cell is unfrozen run this


In [ ]:
# Partial LLM unfreeze — run this when UPDATE_LLM=False
if not UPDATE_LLM:
    model_root = getattr(base_model, "model", base_model)

    encoder = getattr(model_root, "encoder", None)
    projector = getattr(model_root, "projector", None)
    language_model = getattr(model_root, "language_model", None)

    if encoder is not None:
        encoder.requires_grad_(UPDATE_ENCODER)
    if projector is not None:
        projector.requires_grad_(UPDATE_PROJECTOR)
    if language_model is not None:
        language_model.requires_grad_(False)

    if language_model is None:
        raise AttributeError("Could not find the language model submodule for partial unfreezing.")

    lm_body = getattr(language_model, "model", language_model)
    lm_layers = getattr(lm_body, "layers", None)
    if lm_layers is None and hasattr(lm_body, "transformer"):
        lm_layers = getattr(lm_body.transformer, "h", None)

    if lm_layers is None:
        raise AttributeError("Could not locate LLM layers for partial unfreezing.")

    for layer in lm_layers[-NUM_LLM_LAYERS_TO_UNFREEZE:]:
        layer.requires_grad_(True)

    print(f'Encoder trainable:    {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}' if encoder is not None else 'Encoder trainable:    N/A')
    print(f'Projector trainable:  {sum(p.numel() for p in projector.parameters() if p.requires_grad):,}' if projector is not None else 'Projector trainable:  N/A')
    print(f'LLM trainable:        {sum(p.numel() for p in language_model.parameters() if p.requires_grad):,}' if language_model is not None else 'LLM trainable:        N/A')


In [21]:
# Additional safety for generation
if hasattr(base_model, "generation_config"):
    base_model.generation_config.pad_token_id = tokenizer.pad_token_id
    base_model.generation_config.eos_token_id = tokenizer.eos_token_id

## Configure Trainer

In [22]:
# Fall back to defaults if the settings cell was skipped
OUTPUT_DIR = globals().get('OUTPUT_DIR', os.path.join('/jupyter_kernel', 'trained_models', 'ibm-granite-speech-kenyan-english-nonstandard-robust_v1_run1'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOGGING_STEPS = globals().get('LOGGING_STEPS', 5)
SAVE_STEPS = globals().get('SAVE_STEPS', 50)
MAX_EPOCHS = globals().get('MAX_EPOCHS', 10)
MAX_STEPS = globals().get('MAX_STEPS', 2000)
LEARNING_RATE = globals().get('LEARNING_RATE', 1e-6)
LR_SCHEDULER_TYPE = globals().get('LR_SCHEDULER_TYPE', 'polynomial')
LR_WARMUP_STEPS = globals().get('LR_WARMUP_STEPS', 100)
LR_END = globals().get('LR_END', 1e-8)
LR_DECAY_POWER = globals().get('LR_DECAY_POWER', 1)
WEIGHT_DECAY = globals().get('WEIGHT_DECAY', 0.01)
EARLY_STOPPING_PATIENCE = globals().get('EARLY_STOPPING_PATIENCE', 7)
BATCH_SIZE = globals().get('BATCH_SIZE', 8)
EVAL_BATCH_SIZE = globals().get('EVAL_BATCH_SIZE', 8)

MAX_GEN_LEN = globals().get('MAX_GEN_LEN', 256)
EVAL_ON_START = globals().get('EVAL_ON_START', True)
EVAL_STEPS = globals().get('EVAL_STEPS', 50)
USE_FP16 = globals().get('USE_FP16', False)
USE_BF16 = globals().get('USE_BF16', True)
NUM_CHECKPOINTS_TO_STORE = globals().get('NUM_CHECKPOINTS_TO_STORE', 2)

from transformers import GenerationConfig

# Clean GenerationConfig - avoid max_length conflict
# Use max_new_tokens only; keep generation length explicit without conflicting max_length.
generation_config = GenerationConfig(
    max_new_tokens=MAX_GEN_LEN,
    do_sample=False,
    num_beams=1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=LOGGING_STEPS,
    report_to=['tensorboard'],
    fp16=USE_FP16,
    bf16=USE_BF16,
    push_to_hub=False,
    remove_unused_columns=False,
    num_train_epochs=MAX_EPOCHS,
    max_steps=MAX_STEPS,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    per_device_train_batch_size=BATCH_SIZE,
    eval_on_start=EVAL_ON_START,
    predict_with_generate=True,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    eval_steps=EVAL_STEPS,
    eval_strategy='steps',
    generation_max_length=MAX_GEN_LEN,
    generation_config=generation_config,
    metric_for_best_model='wer',
    greater_is_better=False,
    load_best_model_at_end=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    lr_scheduler_kwargs={'lr_end': LR_END, 'power': LR_DECAY_POWER},
    learning_rate=LEARNING_RATE,
    warmup_steps=LR_WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    save_steps=SAVE_STEPS,
    save_strategy='steps',
    save_total_limit=NUM_CHECKPOINTS_TO_STORE,
)

print(f'Output: {OUTPUT_DIR}')
print(f'LR: {LEARNING_RATE} | Patience: {EARLY_STOPPING_PATIENCE} | Max steps: {MAX_STEPS}')
print(f'Generation max_new_tokens: {MAX_GEN_LEN}')

Output: /jupyter_kernel/trained_models/ibm-granite-speech-4.1-2b-kenyan-english-nonstandard-robust_v1_run1
LR: 1e-06 | Patience: 7 | Max steps: 1000
Generation max_new_tokens: 256


In [23]:
data_collator = DataCollatorGraniteSpeech(
    processor=processor,
    pad_token_id=tokenizer.pad_token_id,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=base_model,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
print('Trainer ready.')

[RANK 0] Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Trainer ready.


## Run Training

In [24]:
# Force correct padding for decoder-only model
tokenizer.padding_side = 'left'
print(f"✅ Tokenizer padding_side set to: {tokenizer.padding_side}")

# Also set it on the processor if it has a tokenizer
if hasattr(processor, 'tokenizer'):
    processor.tokenizer.padding_side = 'left'

✅ Tokenizer padding_side set to: left


In [ ]:
# GraniteSpeech does not support gradient checkpointing in this Transformers version

trainer.args.gradient_checkpointing = False

print("Starting training...")
trainer.train()

# To resume from last checkpoint if interrupted:
# trainer.train(resume_from_checkpoint=True)



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100257, 'bos_token_id': 100257, 'pad_token_id': 100256}.


Starting training...


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step,Training Loss,Validation Loss,Wer,Cer
0,No log,1.943511,0.638172,0.684607
50,7.933000,1.998506,0.620526,0.669698
100,8.082300,1.913852,0.635253,0.681578
150,6.917600,1.777096,0.633670,0.681249


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializ

WER: 0.6382 | CER: 0.6846


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializ

WER: 0.6205 | CER: 0.6697


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializ

WER: 0.6337 | CER: 0.6812


## Evaluate

Clean audio only. These are the reporting numbers for the run report.

In [ ]:
print('--- Dev set (clean) ---')
trainer.evaluate(dev_dataset)

In [ ]:
print('--- Test set (clean) ---')
trainer.evaluate(test_dataset)

## Evaluate on Noise-Augmented Test Set

Reload raw test split and run through the augmentation pipeline.
Compare against clean test WER to quantify the robustness gap.

In [ ]:
noisy_test_dataset = load_dataset_split(DATASET_NAME, split='test', limit_to_30_seconds=True)
noisy_test_dataset = noisy_test_dataset.map(
    prepare_features_augmented,
    remove_columns=['audio'],
    writer_batch_size=1,
    num_proc=num_proc,
)
print(f'Noisy test examples: {len(noisy_test_dataset)}')

print('--- Test set (noise-augmented) ---')
trainer.evaluate(noisy_test_dataset)

## Save Best Model

In [ ]:
best_model_dir = os.path.join(OUTPUT_DIR, 'best_model')
print(f'Saving to: {best_model_dir}')
trainer.model.save_pretrained(best_model_dir, safe_serialization=True)
processor.save_pretrained(best_model_dir)
print('Done.')

In [ ]:
import inspect
print(inspect.getsource(DataCollatorGraniteSpeech))

In [ ]:
# Diagnostic: inspect GraniteSpeech processor audio and mask shapes
import numpy as np
from datasets import load_dataset

sample = train_dataset[0]
print('input_features shape:', np.asarray(sample['input_features']).shape)
print('input_features_mask shape:', np.asarray(sample['input_features_mask']).shape)
print('input_features_mask sample:', np.asarray(sample['input_features_mask'])[:20])
print('input_ids len:', len(sample['input_ids']))
print('labels len:', len(sample['labels']))

# Count audio token IDs inside tokenized prompt
input_ids = np.asarray(sample['input_ids'], dtype=np.int64)
audio_token_id = tokenizer.audio_token_id if hasattr(tokenizer, 'audio_token_id') else tokenizer.convert_tokens_to_ids('<|audio|>')
print('audio_token_id:', audio_token_id)
print('audio token count in input_ids:', int((input_ids == audio_token_id).sum()))
print('prompt sample tokens:', tokenizer.convert_ids_to_tokens(input_ids[:20]))

# Inspect processor expansion on raw audio
raw_example = load_dataset(DATASET_NAME, split='train[:1]')[0]
raw_audio = raw_example['audio']['array']
raw_sr = raw_example['audio']['sampling_rate']
audio_out = processor(text=_ASR_PROMPT, audio=raw_audio, return_tensors=None)
proc_ids = np.asarray(audio_out['input_ids'][0], dtype=np.int64)
print('processor output input_ids len:', len(proc_ids))
print('processor output audio token count:', int((proc_ids == audio_token_id).sum()))
print('processor output first 60 tokens:', tokenizer.convert_ids_to_tokens(proc_ids[:60]))
print('processor output input_features shape:', np.asarray(audio_out['input_features'][0]).shape)
print('processor output input_features_mask shape:', np.asarray(audio_out['input_features_mask'][0]).shape)
print('processor output input_features_mask sum:', int(np.asarray(audio_out['input_features_mask'][0]).sum()))


In [ ]:
# Diagnostic: inspect processor input_ids, audio token counts, and mask behavior
from datasets import load_dataset
import numpy as np

raw_example = load_dataset(DATASET_NAME, split='train[:1]')[0]
raw_audio = raw_example['audio']['array']
raw_sr = raw_example['audio']['sampling_rate']
print('raw_audio shape', np.asarray(raw_audio).shape, 'sr', raw_sr)

audio_out = processor(text=_ASR_PROMPT, audio=raw_audio, return_tensors=None)
proc_ids = np.asarray(audio_out['input_ids'][0], dtype=np.int64)
audio_token_id = tokenizer.audio_token_id if hasattr(tokenizer, 'audio_token_id') else tokenizer.convert_tokens_to_ids('<|audio|>')
print('processor output input_ids len:', len(proc_ids))
print('processor audio token count:', int((proc_ids == audio_token_id).sum()))
print('processor first 100 tokens:', tokenizer.convert_ids_to_tokens(proc_ids[:100]))
print('processor output input_features shape:', np.asarray(audio_out['input_features'][0]).shape)
print('processor output input_features_mask shape:', np.asarray(audio_out['input_features_mask'][0]).shape)
print('processor output input_features_mask sum:', int(np.asarray(audio_out['input_features_mask'][0]).sum()))

# Inspect stored dataset sample token counts too
sample = train_dataset[0]
stored_ids = np.asarray(sample['input_ids'], dtype=np.int64)
print('\nstored sample input_ids len:', len(stored_ids))
print('stored sample audio token count:', int((stored_ids == audio_token_id).sum()))
print('stored sample input_features_mask shape:', np.asarray(sample['input_features_mask']).shape)
print('stored sample input_features_mask sum:', int(np.asarray(sample['input_features_mask']).sum()))


In [ ]:
# Diagnostic: compare processor outputs with/without sampling_rate
sample_audio = sample['input_features'] if False else None
# use raw audio from train_dataset if available
if 'audio' in train_dataset.column_names:
    raw_audio = train_dataset[0]['audio']['array']
    raw_sr = train_dataset[0]['audio']['sampling_rate']
else:
    raw_audio = audio_array
    raw_sr = sample.get('sampling_rate', None)

print('raw_sr', raw_sr)

try:
    out1 = processor(text=_ASR_PROMPT, audio=raw_audio, return_tensors=None)
    print('out1 keys', list(out1.keys()))
    print('out1 input_features shape', np.asarray(out1['input_features'][0]).shape)
    print('out1 mask shape', np.asarray(out1['input_features_mask'][0]).shape)
except Exception as e:
    print('out1 failed', e)

try:
    out2 = processor(text=_ASR_PROMPT, audio=raw_audio, sampling_rate=raw_sr, return_tensors=None)
    print('out2 keys', list(out2.keys()))
    print('out2 input_features shape', np.asarray(out2['input_features'][0]).shape)
    print('out2 mask shape', np.asarray(out2['input_features_mask'][0]).shape)
except Exception as e:
    print('out2 failed', e)
